# 1. Import Library
Memasukkan semua library yang dibutuhkan untuk tahap EDA, Preprocessing, Modeling, dan Evaluasi.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, classification_report

import warnings
warnings.filterwarnings('ignore')

# 2. EDA & Preprocessing

### Load Dataset
Sesuai ketentuan, data **train** hanya digunakan untuk training (termasuk sebagai patokan imputasi dan scaling), dan data **test** khusus digunakan untuk evaluasi akhir. Tidak ada split data lagi di sini.

*(Catatan: Untuk menghindari RAM laptop penuh / Kernel Crash akibat memuat 7,5 juta baris sekaligus, kita langsung mensampling data saat awal load)*

In [2]:
# Mengambil data yang sudah di-split sebelumnya
train_df = pd.read_csv('../dataset/data/train.csv')
test_df = pd.read_csv('../dataset/data/test.csv')

# ================== MEMORY OPTIMIZATION ==================
# Agar laptop tidak crash (Out of Memory), kita ambil 
# 300.000 sampel acak untuk train, dan 50.000 untuk test.
train_limit = min(300000, len(train_df))
test_limit = min(50000, len(test_df))
train_df = train_df.sample(n=train_limit, random_state=42).reset_index(drop=True)
test_df = test_df.sample(n=test_limit, random_state=42).reset_index(drop=True)
# =========================================================

print("Shape of train_df:", train_df.shape)
print("Shape of test_df:", test_df.shape)
train_df.head()

Shape of train_df: (300000, 46)
Shape of test_df: (50000, 46)


,ID,Source,Severity,Start_Time,End_Time,Start_Lat,Start_Lng,End_Lat,End_Lng,Distance(mi),...,Roundabout,Station,Stop,Traffic_Calming,Traffic_Signal,Turning_Loop,Sunrise_Sunset,Civil_Twilight,Nautical_Twilight,Astronomical_Twilight
0,A-5220458,Source1,2,2022-08-26 16:45:19,2022-08-27 17:43:40,36.147841,-95.965453,36.147836,-95.964697,0.042,...,False,False,True,False,False,False,Day,Day,Day,Day
1,A-910157,Source2,2,2021-09-07 05:26:59,2021-09-07 07:18:00,35.031857,-78.875946,NaN,NaN,0.000,...,False,False,False,False,False,False,Night,Night,Night,Day
2,A-3898300,Source1,2,2022-06-21 17:31:50,2022-06-21 18:52:31,29.930587,-90.079140,29.931243,-90.079700,0.056,...,False,False,False,False,False,False,Day,Day,Day,Day
3,A-7633280,Source1,4,2018-01-16 04:13:45,2018-01-16 10:13:45,40.840531,-73.281480,40.837230,-73.281200,0.229,...,False,False,False,False,False,False,Night,Night,Night,Night
4,A-3183102,Source2,2,2017-11-13 09:07:30,2017-11-13 09:37:14,31.693895,-106.333054,NaN,NaN,0.000,...,False,False,False,False,False,False,Day,Day,Day,Day


### Binarisasi Severity
Mengubah target Severity dari 4 kelas (1,2,3,4) menjadi 2 kelas: 0 (Severity 1-2) dan 1 (Severity 3-4).

In [3]:
if train_df['Severity'].isin([0, 1]).all():
    print("Severity sudah dalam format biner (0/1), tidak perlu binarisasi ulang.")
else:
    train_df['Severity'] = train_df['Severity'].map({1:0, 2:0, 3:1, 4:1})
    test_df['Severity'] = test_df['Severity'].map({1:0, 2:0, 3:1, 4:1})
    print("Binarisasi Severity selesai (1,2→0; 3,4→1).")

print("Distribusi kelas:")
print("Train:", train_df['Severity'].value_counts().to_dict())
print("Test:", test_df['Severity'].value_counts().to_dict())

Binarisasi Severity selesai (1,2→0; 3,4→1).
Distribusi kelas:
Train: {0: 241770, 1: 58230}
Test: {0: 40343, 1: 9657}


### Drop Kolom ID
Menghapus kolom ID karena tidak relevan sebagai fitur.

In [4]:
train_df = train_df.drop(columns=['ID'])
test_df = test_df.drop(columns=['ID'])
print(f"Shape after dropping ID: train {train_df.shape}, test {test_df.shape}")

Shape after dropping ID: train (300000, 45), test (50000, 45)


In [5]:
# Cek missing value SEBELUM imputasi
na_train = train_df.isnull().sum().sum()
na_test = test_df.isnull().sum().sum()
print(f"Missing value sebelum imputasi - train: {na_train}, test: {na_test}")
if na_train > 0 or na_test > 0:
    print("Akan diisi median (numerik) & modus (kategorikal) di tahap 2.1")

Missing value sebelum imputasi - train: 497943, test: 82872
Akan diisi median (numerik) & modus (kategorikal) di tahap 2.1


### 2.1 Missing Value Handling
Penanganan missing value dilakukan dengan mengisi kekosongan (imputasi). Kolom numerik diisi dengan **median**, sedangkan kolom kategorikal diisi dengan **modus**. Ingat, perhitungan median/modus **hanya** didapatkan dari data train.

In [6]:
print("Missing values di data train:\n", train_df.isnull().sum())

# Tentukan mana kolom numerik dan kategorikal
numeric_cols = train_df.select_dtypes(include=['float64', 'int64']).columns
categorical_cols = train_df.select_dtypes(include=['object', 'category']).columns

# Simpan nilai imputasi untuk deployment
median_impute = {}
for col in numeric_cols:
    median_val = train_df[col].median()
    median_impute[col] = median_val if not pd.isna(median_val) else 0
    train_df[col] = train_df[col].fillna(median_val)
    test_df[col] = test_df[col].fillna(median_val)

mode_impute = {}
for col in categorical_cols:
    mode_val = train_df[col].mode()[0]
    mode_impute[col] = mode_val if not train_df[col].mode().empty else 'unknown'
    train_df[col] = train_df[col].fillna(mode_val)
    test_df[col] = test_df[col].fillna(mode_val)

# Verifikasi missing value setelah imputasi
remaining_na_train = train_df.isnull().sum().sum()
remaining_na_test = test_df.isnull().sum().sum()
print(f"Missing value setelah imputasi - train: {remaining_na_train}, test: {remaining_na_test}")
if remaining_na_train == 0 and remaining_na_test == 0:
    print("✅ Tidak ada missing value tersisa. Data siap diproses ke tahap selanjutnya.")
else:
    print("⚠️  Masih ada missing value!")
    print("Train:\n", train_df.columns[train_df.isnull().any()].tolist())
    print("Test:\n", test_df.columns[test_df.isnull().any()].tolist())

# Tampilkan data setelah imputasi
from IPython.display import display
print("\nData train setelah imputasi (sampel):")
display(train_df.head(10))
print(f"Shape: {train_df.shape}")

Missing values di data train:
 Source                        0
Severity                      0
Start_Time                    0
End_Time                      0
Start_Lat                     0
Start_Lng                     0
End_Lat                  131936
End_Lng                  131936
Distance(mi)                  0
Description                   0
Street                      431
City                          8
County                        0
State                         0
Zipcode                      70
Country                       0
Timezone                    280
Airport_Code                830
Weather_Timestamp          4634
Temperature(F)             6301
Wind_Chill(F)             77756
Humidity(%)                6692
Pressure(in)               5452
Visibility(mi)             6810
Wind_Direction             6778
Wind_Speed(mph)           22061
Precipitation(in)         85681
Weather_Condition          6695
Amenity                       0
Bump                          0
Crossing 

,Source,Severity,Start_Time,End_Time,Start_Lat,Start_Lng,End_Lat,End_Lng,Distance(mi),Description,...,Roundabout,Station,Stop,Traffic_Calming,Traffic_Signal,Turning_Loop,Sunrise_Sunset,Civil_Twilight,Nautical_Twilight,Astronomical_Twilight
0,Source1,0,2022-08-26 16:45:19,2022-08-27 17:43:40,36.147841,-95.965453,36.147836,-95.964697,0.042,Construction on 11TH ST near HOUSE 1809 Expect...,...,False,False,True,False,False,False,Day,Day,Day,Day
1,Source2,0,2021-09-07 05:26:59,2021-09-07 07:18:00,35.031857,-78.875946,36.179155,-88.003069,0.000,Right hand shoulder blocked due to accident on...,...,False,False,False,False,False,False,Night,Night,Night,Day
2,Source1,0,2022-06-21 17:31:50,2022-06-21 18:52:31,29.930587,-90.079140,29.931243,-90.079700,0.056,Incident on PHILIP ST near HOUSE 1313 Drive wi...,...,False,False,False,False,False,False,Day,Day,Day,Day
3,Source1,1,2018-01-16 04:13:45,2018-01-16 10:13:45,40.840531,-73.281480,40.837230,-73.281200,0.229,Closed at RT-454 - Road closed due to accident.,...,False,False,False,False,False,False,Night,Night,Night,Night
4,Source2,0,2017-11-13 09:07:30,2017-11-13 09:37:14,31.693895,-106.333054,36.179155,-88.003069,0.000,Accident on TX-20 Alameda Ave at Snelson Dr.,...,False,False,False,False,False,False,Day,Day,Day,Day
5,Source1,0,2020-12-18 14:03:00,2020-12-18 16:24:59,36.669605,-119.457510,36.658525,-119.457510,0.766,Incident on S REED AVE near E GOODFELLOW AVE E...,...,False,False,False,False,False,False,Day,Day,Day,Day
6,Source1,0,2021-08-13 07:56:00,2021-08-13 09:15:08,33.551747,-112.237812,33.551961,-112.237819,0.015,Incident on N 83RD AVE near W NORTHERN AVE Exp...,...,False,False,False,False,True,False,Day,Day,Day,Day
7,Source1,1,2023-02-14 08:09:30.000000000,2023-02-14 10:18:35.000000000,38.859240,-77.371314,38.858141,-77.371482,0.076,On Rt. 7700 in the County of Fairfax in the vi...,...,False,False,False,False,True,False,Day,Day,Day,Day
8,Source2,0,2021-06-09 07:54:47,2021-06-09 09:31:41,26.547220,-81.804268,36.179155,-88.003069,0.000,Two lanes blocked due to accident on Daniels P...,...,False,False,False,False,True,False,Day,Day,Day,Day
9,Source2,0,2020-11-20 10:21:44,2020-11-20 12:18:38,35.420769,-97.441277,36.179155,-88.003069,0.000,Accident on 44th St at Sunnylane Rd.,...,False,False,False,False,True,False,Day,Day,Day,Day


Shape: (300000, 45)


In [7]:
t = train_df
ts = test_df

print("Train:\n", t.isnull().sum())
print("Test:\n", ts.isnull().sum())

Train:
 Source                   0
Severity                 0
Start_Time               0
End_Time                 0
Start_Lat                0
Start_Lng                0
End_Lat                  0
End_Lng                  0
Distance(mi)             0
Description              0
Street                   0
City                     0
County                   0
State                    0
Zipcode                  0
Country                  0
Timezone                 0
Airport_Code             0
Weather_Timestamp        0
Temperature(F)           0
Wind_Chill(F)            0
Humidity(%)              0
Pressure(in)             0
Visibility(mi)           0
Wind_Direction           0
Wind_Speed(mph)          0
Precipitation(in)        0
Weather_Condition        0
Amenity                  0
Bump                     0
Crossing                 0
Give_Way                 0
Junction                 0
No_Exit                  0
Railway                  0
Roundabout               0
Station             

### 2.2 Outliers Handling
Kita menggunakan metode IQR (Interquartile Range) untuk membatasi (capping) nilai outlier agar tidak merusak model. PENTING: Kolom target dilarang keras untuk dicapping!

In [8]:
iqr_bounds = {}
for col in numeric_cols:
    if col == 'Severity':
        continue
        
    Q1 = train_df[col].quantile(0.25)
    Q3 = train_df[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    iqr_bounds[col] = {'lower': lower_bound, 'upper': upper_bound}
    
    train_df[col] = np.where(train_df[col] < lower_bound, lower_bound, train_df[col])
    train_df[col] = np.where(train_df[col] > upper_bound, upper_bound, train_df[col])
    
    test_df[col] = np.where(test_df[col] < lower_bound, lower_bound, test_df[col])
    test_df[col] = np.where(test_df[col] > upper_bound, upper_bound, test_df[col])

### 2.3 Encoding
Mengubah data string (teks) menjadi angka numerik yang bisa dipahami model menggunakan `LabelEncoder`. Dibuat secepat mungkin menggunakan dictionary mapping.

In [9]:
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    train_df[col] = le.fit_transform(train_df[col].astype(str))
    
    le_dict = dict(zip(le.classes_, range(len(le.classes_))))
    unknown_val = len(le.classes_)
    label_encoders[col] = {
        'classes': le.classes_.tolist(),
        'mapping': le_dict
    }
    
    le.classes_ = np.append(le.classes_, '<unknown>')
    
    test_df[col] = test_df[col].astype(str).map(le_dict).fillna(unknown_val).astype(int)

### 2.4 Transformasi (Scaling Fitur)
Melakukan standardisasi (rata-rata=0, std=1) agar model yang sensitif pada jarak bekerja lebih optimal.

In [10]:
# TARGET KOLOM DISET KE 'Severity'
TARGET_KOLOM = 'Severity' 

try:
    # Memisahkan Fitur (X) dan Target Label (y)
    X_train_raw = train_df.drop(columns=[TARGET_KOLOM])
    y_train = train_df[TARGET_KOLOM]
    
    X_test_raw = test_df.drop(columns=[TARGET_KOLOM])
    y_test = test_df[TARGET_KOLOM]
    
    # Inisiasi Scaler
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_raw)
    X_test_scaled = scaler.transform(X_test_raw) # Scaling test pakai patokan train
except KeyError:
    print("WARNING: Jangan lupa ubah variabel TARGET_KOLOM di atas dengan nama kolom yang benar di dataset ini.")

# 3. Modeling
Membuat model klasifikasi. Kita akan menggunakan algoritma **Random Forest Classifier** karena secara umum algoritma ini tangguh (robust) terhadap struktur data yang belum linear sempurna dan memberikan akurasi yang solid.

In [11]:
try:
    # Menggunakan class_weight='balanced' untuk mengatasi imbalanced data (kelas 1 dan 4 yang langka)
    # Karena di Langkah 2 data sudah di-sample, datanya sudah teracak dengan baik.
    rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1, class_weight='balanced')
    
    print(f"Melatih model Random Forest (Balanced) pada seluruh train_df ({len(X_train_scaled)} baris)...")
    rf_model.fit(X_train_scaled, y_train)
    print("Model Random Forest berhasil dilatih!")
except NameError:
    print("Harap lengkapi tahap 2.4 terlebih dahulu.")

Melatih model Random Forest (Balanced) pada seluruh train_df (300000 baris)...
Model Random Forest berhasil dilatih!


# 4. Evaluasi (Test)
Menggunakan fitur dari data uji (test.csv) ke dalam model, lalu membandingkan hasil prediksinya dengan label aslinya untuk mencari nilai Recall, Precision, dan Accuracy.

In [12]:
try:
    print(f"Menguji data test...")
    y_pred = rf_model.predict(X_test_scaled)
    
    # Evaluasi Metriks
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    
    print("=== HASIL EVALUASI MODEL ===")
    print(f"Akurasi   (Accuracy)  : {acc:.4f}")
    print(f"Presisi   (Precision) : {prec:.4f}")
    print(f"Recall    (Recall)    : {rec:.4f}")
    
    # Menampilkan report lebih lengkap (opsional)
    print("\nClassification Report Lengkap:\n")
    print(classification_report(y_test, y_pred, zero_division=0))
except NameError:
    print("Harap lengkapi tahap sebelumnya.")

Menguji data test...
=== HASIL EVALUASI MODEL ===
Akurasi   (Accuracy)  : 0.8471
Presisi   (Precision) : 0.8392
Recall    (Recall)    : 0.8471

Classification Report Lengkap:

              precision    recall  f1-score   support

           0       0.89      0.92      0.91     40343
           1       0.62      0.53      0.57      9657

    accuracy                           0.85     50000
   macro avg       0.76      0.73      0.74     50000
weighted avg       0.84      0.85      0.84     50000



# 5. Simpan Model & Artifacts
Menyimpan model yang sudah dilatih beserta preprocessing artifacts ke folder `model/` untuk digunakan di deployment.

In [13]:
import joblib
import os

model_dir = '../model'
os.makedirs(model_dir, exist_ok=True)

feature_names = X_train_raw.columns.tolist()

preprocessing = {
    'numeric_cols': numeric_cols,
    'categorical_cols': categorical_cols,
    'target_column': TARGET_KOLOM,
    'median_impute': median_impute,
    'mode_impute': mode_impute,
    'iqr_bounds': iqr_bounds,
    'label_encoders': label_encoders,
    'feature_names': feature_names
}

joblib.dump(rf_model, os.path.join(model_dir, 'random_forest.pkl'))
joblib.dump(scaler, os.path.join(model_dir, 'scaler.pkl'))
joblib.dump(preprocessing, os.path.join(model_dir, 'preprocessing.pkl'))

print(f"Model dan artifacts berhasil disimpan ke folder '{model_dir}/'")
print("  - random_forest.pkl")
print("  - scaler.pkl")
print("  - preprocessing.pkl")

Model dan artifacts berhasil disimpan ke folder '../model/'
  - random_forest.pkl
  - scaler.pkl
  - preprocessing.pkl
